# Inference Examples - VariantClassifier

Este notebook demonstra como usar o modelo treinado para inferência.

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import sys
sys.path.append('../src')

from modeling.preprocessing import VariantPreprocessor

print('Bibliotecas importadas com sucesso!')

## 1. Carregar Modelo Treinado

In [ ]:
# Carregar modelo e preprocessor
from modeling.ensemble import VariantClassifierEnsemble

model = VariantClassifierEnsemble.load('../models/variant_ensemble.joblib')
preprocessor = VariantPreprocessor.load('../models/preprocessor.joblib')

print('Modelo e preprocessor carregados!')
print(f'\nModelo configurado para {len(model.classes)} classes:')
print(model.classes)

## 2. Inferência em Uma Variante

In [ ]:
# Exemplo 1: Variante patogênica conhecida (BRCA1)
variant_pathogenic = {
    'chromosome': 'chr17',
    'position': 43044295,
    'gene': 'BRCA1',
    'variant_type': 'missense_variant',
    'ref': 'G',
    'alt': 'A',
    'allele_frequency': 0.0,
    'allele_frequency_popmax': 0.0,
    'gnomad_homozygotes': 0,
    'revel_score': 0.94,
    'cadd_phred': 35.0,
    'spliceai_delta': 0.1,
    'mutpred_score': 0.89,
    'eigen_cadd': 0.95,
    'mvp_score': 0.92,
    'metasvm_score': 0.88,
    'consequence': 'missense_variant',
    'exon_number': 10,
    'transcript': 'NM_007294.3',
    'pvs1_lof': False,
    'ps1_same_aa': False,
    'ps2_de_novo': False,
    'ps3_functional_well_established': True,
    'ps4_case_controls': False,
    'pm1_hotspot': True,
    'pm2_rare': True,
    'pm3_trans_pathogenic': False,
    'pm4_length': False,
    'pm5_different_change': False,
    'pm6_de_novo_unconfirmed': False,
    'pp1_cosegregation': False,
    'pp2_missense_gene': True,
    'pp3_multiple_computational': True,
    'pp4_phenotype': False,
    'pp5_reputable_source': False,
    'ba1_frequency': False,
    'bs1_frequency': False,
    'bs2_healthy_adult': False,
    'bs3_functional_benign': False,
    'bs4_case_controls': False,
    'bp1_misense_gene': False,
    'bp2_trans_benign': False,
    'bp3_repetitive_region': False,
    'bp4_multiple_computational_benign': False,
    'bp5_different_mechanism': False,
    'bp6_reputable_benign': False,
    'bp7_synonymous': False,
    'gene_haploinsufficient': True,
    'gene_triplosensitive': False,
    'gene_constraint_lof': 0.99,
    'gene_constraint_missense': 0.95,
    'protein_domain': 'BRCT',
    'clinical_significance': 'Pathogenic',
    'sift_score': 0.0,
    'polyphen_score': 0.95,
    'mutationtaster_score': 1.0
}

# Converter para DataFrame
var_df = pd.DataFrame([variant_pathogenic])

# Preprocessar
var_proc = preprocessor.transform(var_df)

# Predizer
prediction = model.predict(var_proc)[0]
probabilities = model.predict_proba(var_proc)[0]
confidence = probabilities.max()

print('Variante BRCA1 (missense, REVEL=0.94):')
print(f'Classificação: {prediction}')
print(f'Confiança: {confidence:.2%}')
print(f'\nProbabilidades:')
for cls, prob in zip(model.classes, probabilities):
    print(f'  {cls}: {prob:.4f}')

In [ ]:
# Exemplo 2: Variante benigna
variant_benign = {
    'chromosome': 'chr13',
    'position': 32906830,
    'gene': 'BRCA2',
    'variant_type': 'synonymous_variant',
    'ref': 'G',
    'alt': 'A',
    'allele_frequency': 0.002,
    'allele_frequency_popmax': 0.003,
    'gnomad_homozygotes': 10,
    'revel_score': 0.05,
    'cadd_phred': 5.0,
    'spliceai_delta': 0.0,
    'mutpred_score': 0.12,
    'eigen_cadd': 0.1,
    'mvp_score': 0.08,
    'metasvm_score': 0.05,
    'consequence': 'synonymous_variant',
    'exon_number': 5,
    'transcript': 'NM_000059.3',
    'pvs1_lof': False,
    'ps1_same_aa': False,
    'ps2_de_novo': False,
    'ps3_functional_well_established': False,
    'ps4_case_controls': False,
    'pm1_hotspot': False,
    'pm2_rare': False,
    'pm3_trans_pathogenic': False,
    'pm4_length': False,
    'pm5_different_change': False,
    'pm6_de_novo_unconfirmed': False,
    'pp1_cosegregation': False,
    'pp2_missense_gene': False,
    'pp3_multiple_computational': False,
    'pp4_phenotype': False,
    'pp5_reputable_source': False,
    'ba1_frequency': False,
    'bs1_frequency': True,
    'bs2_healthy_adult': True,
    'bs3_functional_benign': True,
    'bs4_case_controls': False,
    'bp1_misense_gene': False,
    'bp2_trans_benign': False,
    'bp3_repetitive_region': False,
    'bp4_multiple_computational_benign': True,
    'bp5_different_mechanism': False,
    'bp6_reputable_benign': True,
    'bp7_synonymous': True,
    'gene_haploinsufficient': False,
    'gene_triplosensitive': False,
    'gene_constraint_lof': 0.5,
    'gene_constraint_missense': 0.3,
    'protein_domain': 'None',
    'clinical_significance': 'Benign',
    'sift_score': 1.0,
    'polyphen_score': 0.0,
    'mutationtaster_score': 0.0
}

var_benign_df = pd.DataFrame([variant_benign])
var_benign_proc = preprocessor.transform(var_benign_df)

pred_benign = model.predict(var_benign_proc)[0]
proba_benign = model.predict_proba(var_benign_proc)[0]
conf_benign = proba_benign.max()

print('\nVariante BRCA2 (synonymous, AF=0.002):')
print(f'Classificação: {pred_benign}')
print(f'Confiança: {conf_benign:.2%}')
print(f'\nProbabilidades:')
for cls, prob in zip(model.classes, proba_benign):
    print(f'  {cls}: {prob:.4f}')

## 3. Inferência em Lote

In [ ]:
# Carregar dataset de teste
test_df = pd.read_csv('../data/splits/test.csv')

# Amostra de 10 variantes
sample_variants = test_df.sample(n=10, random_state=42)
X_sample = sample_variants.drop('pathogenicity', axis=1)
y_sample_true = sample_variants['pathogenicity']

# Preprocessar
X_sample_proc = preprocessor.transform(X_sample)

# Predizer
y_pred = model.predict(X_sample_proc)
y_proba = model.predict_proba(X_sample_proc)

# Compilar resultados
results = []
for i in range(len(sample_variants)):
    results.append({
        'Gene': X_sample.iloc[i]['gene'],
        'Posição': X_sample.iloc[i]['position'],
        'Tipo': X_sample.iloc[i]['variant_type'],
        'Real': y_sample_true.iloc[i],
        'Predição': y_pred[i],
        'Confiança': f'{y_proba[i].max():.2%}',
        'Correto': y_pred[i] == y_sample_true.iloc[i]
    })

results_df = pd.DataFrame(results)
display(results_df)

print(f'\nAcurácia na amostra: {results_df["Correto"].mean():.2%}')

## 4. Inferência via API REST

In [ ]:
# Configuração da API
API_URL = 'http://localhost:8000'

# Verificar se API está rodando
try:
    response = requests.get(f'{API_URL}/health', timeout=2)
    if response.status_code == 200:
        print('API está rodando!')
        print(f'Status: {response.json()}')
    else:
        print('API não respondendo corretamente.')
        print('Inicie a API com: bash start_api.sh')
except requests.exceptions.ConnectionError:
    print('API não está rodando.')
    print('Inicie a API com: bash start_api.sh')
except Exception as e:
    print(f'Erro: {e}')

In [ ]:
# Exemplo de requisição via API (se estiver rodando)
try:
    # Preparar payload
    payload = {
        'chromosome': 'chr17',
        'position': 43044295,
        'gene': 'BRCA1',
        'variant_type': 'missense_variant',
        'ref': 'G',
        'alt': 'A',
        'allele_frequency': 0.0,
        'allele_frequency_popmax': 0.0,
        'gnomad_homozygotes': 0,
        'revel_score': 0.94,
        'cadd_phred': 35.0,
        'spliceai_delta': 0.1,
        'mutpred_score': 0.89,
        'eigen_cadd': 0.95,
        'mvp_score': 0.92,
        'metasvm_score': 0.88,
        'consequence': 'missense_variant',
        'exon_number': 10,
        'transcript': 'NM_007294.3',
        'pvs1_lof': False,
        'ps1_same_aa': False,
        'ps2_de_novo': False,
        'ps3_functional_well_established': True,
        'ps4_case_controls': False,
        'pm1_hotspot': True,
        'pm2_rare': True,
        'pm3_trans_pathogenic': False,
        'pm4_length': False,
        'pm5_different_change': False,
        'pm6_de_novo_unconfirmed': False,
        'pp1_cosegregation': False,
        'pp2_missense_gene': True,
        'pp3_multiple_computational': True,
        'pp4_phenotype': False,
        'pp5_reputable_source': False,
        'ba1_frequency': False,
        'bs1_frequency': False,
        'bs2_healthy_adult': False,
        'bs3_functional_benign': False,
        'bs4_case_controls': False,
        'bp1_misense_gene': False,
        'bp2_trans_benign': False,
        'bp3_repetitive_region': False,
        'bp4_multiple_computational_benign': False,
        'bp5_different_mechanism': False,
        'bp6_reputable_benign': False,
        'bp7_synonymous': False,
        'gene_haploinsufficient': True,
        'gene_triplosensitive': False,
        'gene_constraint_lof': 0.99,
        'gene_constraint_missense': 0.95,
        'protein_domain': 'BRCT',
        'clinical_significance': 'Pathogenic',
        'sift_score': 0.0,
        'polyphen_score': 0.95,
        'mutationtaster_score': 1.0
    }
    
    # Fazer requisição
    response = requests.post(f'{API_URL}/predict', json=payload, timeout=5)
    
    if response.status_code == 200:
        result = response.json()
        print('Predição via API:')
        print(f'Classificação: {result["classification"]}')
        print(f'Confiança: {result["confidence"]:.2%}')
        print(f'\nProbabilidades:')
        for cls, prob in result['probabilities'].items():
            print(f'  {cls}: {prob:.4f}')
    else:
        print(f'Erro na requisição: {response.status_code}')
        print(response.text)
        
except requests.exceptions.ConnectionError:
    print('\nAPI não disponível. Pule esta seção.')
except Exception as e:
    print(f'Erro: {e}')

## 5. Batch Prediction via API

In [ ]:
# Exemplo de batch prediction via API (se estiver rodando)
try:
    # Preparar batch payload
    batch_payload = [variant_pathogenic, variant_benign]
    
    # Fazer requisição
    response = requests.post(f'{API_URL}/predict/batch', json=batch_payload, timeout=10)
    
    if response.status_code == 200:
        results = response.json()
        print('Predições em lote via API:')
        for i, result in enumerate(results['predictions']):
            print(f'\nVariante {i+1}:')
            print(f'  Classificação: {result["classification"]}')
            print(f'  Confiança: {result["confidence"]:.2%}')
    else:
        print(f'Erro na requisição: {response.status_code}')
        print(response.text)
        
except requests.exceptions.ConnectionError:
    print('\nAPI não disponível. Pule esta seção.')
except Exception as e:
    print(f'Erro: {e}')

## 6. Análise de Incerteza

In [ ]:
# Identificar predições com baixa confiança
test_df = pd.read_csv('../data/splits/test.csv')
X_test = test_df.drop('pathogenicity', axis=1)
y_test = test_df['pathogenicity']

# Preprocessar e predizer
X_test_proc = preprocessor.transform(X_test)
y_test_pred = model.predict(X_test_proc)
y_test_proba = model.predict_proba(X_test_proc)

# Calcular confianças
confidences = y_test_proba.max(axis=1)

# Thresholds de confiança
HIGH_CONFIDENCE = 0.90
MEDIUM_CONFIDENCE = 0.70

high_conf = (confidences >= HIGH_CONFIDENCE).sum()
medium_conf = ((confidences >= MEDIUM_CONFIDENCE) & (confidences < HIGH_CONFIDENCE)).sum()
low_conf = (confidences < MEDIUM_CONFIDENCE).sum()

print('Distribuição de Confiança:')
print(f'Alta (>= 90%): {high_conf} ({high_conf/len(test_df)*100:.1f}%)')
print(f'Média (70-90%): {medium_conf} ({medium_conf/len(test_df)*100:.1f}%)')
print(f'Baixa (< 70%): {low_conf} ({low_conf/len(test_df)*100:.1f}%)')

# Mostrar exemplos de baixa confiança
low_conf_indices = np.where(confidences < MEDIUM_CONFIDENCE)[0]
if len(low_conf_indices) > 0:
    print(f'\nExemplos de baixa confiança (primeiros 5):')
    for idx in low_conf_indices[:5]:
        print(f'\nVariante {idx}:')
        print(f'  Gene: {X_test.iloc[idx]["gene"]}')
        print(f'  Real: {y_test.iloc[idx]}')
        print(f'  Predição: {y_test_pred[idx]}')
        print(f'  Confiança: {confidences[idx]:.2%}')
        print(f'  Top 2 probabilidades:')
        sorted_probs = sorted(zip(model.classes, y_test_proba[idx]), key=lambda x: -x[1])
        for cls, prob in sorted_probs[:2]:
            print(f'    {cls}: {prob:.4f}')

## 7. Resumo

In [ ]:
print('\n' + '='*80)
print('RESUMO DE INFERÊNCIA')
print('='*80)

print(f'\nModelo carregado: VariantClassifierEnsemble')
print(f'Classes: {len(model.classes)}')
print(f'Features: {X_test_proc.shape[1]}')

print(f'\nMétodos de inferência disponíveis:')
print(f'  1. Direto: model.predict() / model.predict_proba()')
print(f'  2. API REST: POST /predict')
print(f'  3. Batch API: POST /predict/batch')

print(f'\nRecomendações de uso:')
print(f'  - Alta confiança (>= 90%): Confie na predição')
print(f'  - Média confiança (70-90%): Use com cautela')
print(f'  - Baixa confiança (< 70%): Requer revisão manual')